In [ ]:
import fastf1
import plotly.graph_objects as go
import yaml
import pandas as pd

with open("C:/Users/boncl/Repos/simply-lovely/config/teams_drivers_info.yaml", "r") as file:
    config = yaml.safe_load(file)
fastf1.Cache.enable_cache('C:/Users/boncl/Repos/simply-lovely/fastf1_cache')

compound_colours = {
    "SOFT": "#E10600",
    "MEDIUM": "#FFD12E",
    "HARD": "#FAF9F6",
    "INTERMEDIATE": "#43B02A",
    "WET": "#0067AD"
}

In [ ]:
silverstone_quali_laps = fastf1.get_session(2026, 'Silverstone', 'Q')
silverstone_quali_laps.load()
quali_laps = pd.DataFrame(silverstone_quali_laps.laps.pick_quicklaps())
# quali_laps['LapTime'] = quali_laps['LapTime'].dt.total_seconds()
quali_laps

In [ ]:
silverstone_quali_status = pd.DataFrame(silverstone_quali_laps.session_status)
silverstone_quali_session = quali_laps.merge(silverstone_quali_status, on=['Time'], how='outer')
silverstone_quali_session['Status'] = silverstone_quali_session['Status'].ffill()

In [ ]:
silverstone_quali_session = silverstone_quali_session.copy()

# LapTime is required to draw a rectangle
silverstone_quali_session['LapTime'] = pd.to_timedelta(
    silverstone_quali_session['LapTime'],
    errors='coerce'
)

# LapStartTime is also required
silverstone_quali_session['LapStartTime'] = pd.to_timedelta(
    silverstone_quali_session['LapStartTime'],
    errors='coerce'
)

# Keep only rows that actually represent a timed lap
plot_laps = silverstone_quali_session.dropna(
    subset=['LapTime', 'LapStartTime']
).copy()

plot_laps['LapEndTime'] = (
    plot_laps['LapStartTime'] +
    plot_laps['LapTime']
)

status_changes = (
    silverstone_quali_session[['Time', 'Status']]
    .drop_duplicates()
    .sort_values('Time')
)

status_changes = status_changes[
    status_changes['Status'] != status_changes['Status'].shift()
]

gaps = []

finish_time = None

for _, row in status_changes.iterrows():

    if row['Status'] == 'Finished':
        finish_time = pd.Timedelta(row['Time'])

    elif row['Status'] == 'Started' and finish_time is not None:

        # Laps that started before the official session finished
        session_laps = plot_laps[
            plot_laps['LapStartTime'] <= finish_time
        ]

        # The latest time any of those laps actually finished
        actual_finish_time = session_laps['LapEndTime'].max()

        gaps.append(
            (actual_finish_time, pd.Timedelta(row['Time']))
        )

        finish_time = None

def compress_time(time):

    time = pd.Timedelta(time)

    offset = pd.Timedelta(0)

    for gap_start, gap_end in gaps:

        if time <= gap_start:
            break

        offset += min(time, gap_end) - gap_start

    return time - offset

plot_laps['CompressedLapStartTime'] = (
    plot_laps['LapStartTime']
    .apply(compress_time)
)

# Convert to numeric seconds for Plotly
plot_laps['CompressedLapStartSeconds'] = (
    plot_laps['CompressedLapStartTime']
    .dt.total_seconds()
)

In [ ]:
fig4 = go.Figure()

# Fastest valid qualifying lap
fastest_quali_lap = plot_laps['LapTime'].min()

# Drivers with at least one valid timed lap
drivers = plot_laps['Driver'].unique()

# Numerical y position for each driver
driver_positions = {
    driver: i for i, driver in enumerate(drivers)
}

for driver in drivers:

    driver_quali_laps = plot_laps[
        plot_laps['Driver'] == driver
    ]

    driver_fastest_quali_lap = driver_quali_laps['LapTime'].min()

    y_pos = driver_positions[driver]

    for _, lap in driver_quali_laps.iterrows():

        # Decide rectangle colour
        if lap['LapTime'] == fastest_quali_lap:
            colour = 'purple'

        elif lap['LapTime'] == driver_fastest_quali_lap:
            colour = 'green'

        else:
            colour = 'yellow'

        # Start in compressed session time
        start_time = lap['CompressedLapStartSeconds']

        # LapTime is a Timedelta -> convert to seconds
        lap_duration = lap['LapTime'].total_seconds()

        end_time = start_time + lap_duration

        fig4.add_trace(
            go.Scatter(
                x=[
                    start_time,
                    end_time,
                    end_time,
                    start_time,
                    start_time
                ],

                y=[
                    y_pos - 0.35,
                    y_pos - 0.35,
                    y_pos + 0.35,
                    y_pos + 0.35,
                    y_pos - 0.35
                ],

                mode='lines',
                fill='toself',
                fillcolor=colour,
                line=dict(
                    color=colour,
                    width=1
                ),
                showlegend=False,

                hovertemplate=(
                    f"Driver: {driver}<br>"
                    f"Lap: {lap['LapNumber']}<br>"
                    f"Lap time: {lap['LapTime']}<extra></extra>"
                )
            )
        )

qualifying_starts = (
    status_changes[
        status_changes['Status'] == 'Started'
    ]['Time']
    .apply(pd.Timedelta)
    .tolist()
)

tickvals = [
    compress_time(start).total_seconds()
    for start in qualifying_starts[:3]
]

fig4.update_layout(
    height=700,
    xaxis=dict(
        title='Qualifying Session Time',
        tickmode='array',
        tickvals=tickvals,
        ticktext=[
            'Q1 Start',
            'Q2 Start',
            'Q3 Start'
        ],
        showgrid=True
    ),
    yaxis=dict(
        tickmode='array',
        tickvals=list(driver_positions.values()),
        ticktext=list(driver_positions.keys()),
        autorange='reversed'
    ),
    title = dict(text = "Silverstone Quali Plot", font = dict( size = 20)
))

fig4.show()

In [ ]:
fig5 = go.Figure()
for driver in quali_laps['Driver']:
    driver_laps = quali_laps[quali_laps['Driver'] == driver]
    driver_team = driver_laps['Team'].min()
    driver_colour = config['teams'][driver_team]['drivers'][driver]['colour']
    driver_fastest_quali_lap = driver_laps.laps.pick_driver(driver).pick_fastest()
    telemetry = driver_fastest_quali_lap.get_telemetry()
    fig5.add_trace(x=telemetry['Distance'], y=telemetry['Velocity'], name=driver)

fig5.update_layout(title="Fastest QUali Laps Comparison", xaxis_title="Distance (m)", yaxis_title="Velocity (m/s)")
fig5.show()